# Injected thoughts and verbal introspection — GPT-2 XL

**Question.** If a concept's J-lens direction is added to GPT-2 XL's residual stream while it is
asked whether a thought has been injected, does it name that concept — more than with logit-lens
or random directions of the same size, and not when nothing is injected? This is the paper's
injection protocol ([§ws-report](https://transformer-circuits.pub/2026/workspace/index.html#ws-report),
Figure 7) on the GPT-2 single-token concept set (`data/experiments/verbal-introspection-gpt2.json`).

**Answer.** PENDING (filled in after the run on the V100).

GPT-2 XL is a base model without a chat template and the paper's result comes from Claude models;
the Qwen3-4B run of the same protocol (`notebooks/qwen3-4b/jacobian_lens_prefitted/verbal_introspection.ipynb`)
was negative. A negative result here is therefore the expected outcome, and the controls decide
whether any positive number is introspection or output steering.

## Method

**Prompt.** The dialogue of `intro_prompt` is rendered as plain text (`User:` / `Assistant:` turns
separated by blank lines), with `<|endoftext|>` prepended as BOS. The prefill (`word`:
`... The thought is about the word "`) is teacher-forced as the assistant's reply. Every concept is
one GPT-2 token without a leading space, which is how it follows the open quote, so the reported word
is a single next token.

**Injection** (`jlens.workspace.InjectDirections`). For concept token `t` at layer `l`, the vector is
the unit-normalized `v_t = J_l^T C(gamma * u_t)` (GPT-2 uses LayerNorm, so `C` centers it), scaled by
the row's mean residual norm over the injected positions times `strength`, and added after every block
of the band. Two spans: the protocol's (the tokens of the user's question turn) and, as a contrast, the
whole prompt except BOS. The whole-prompt span includes the readout position, where adding `u_t`'s
direction raises `t`'s logit by construction, so a positive result there is output steering, not report.

**Score.** The rank of the concept token in the next-token distribution at the open quote; median
reciprocal rank vs strength (the paper's figure), and the fractions of trials with rank 1 and rank ≤ 5.
`strength = 0` is the control. It is one clean pass shared by every concept, band and mode.

**Controls.** `logit` directions (`J_l = I`) and `random` directions (a fixed Gaussian vector per
concept and layer, seed 0), norm-matched by construction because every direction is unit-normalized
before scaling.

**Uncertainty** (`jlens.bootstrap.introspection_bootstrap`, as `swap_bootstrap` for the swaps). The
resampling unit is the concept: 10,000 resamples of the 101 concepts with replacement, with the J,
logit and random arms and the strength-0 control of the same concept kept together, so the paired
differences J − logit, J − random and J − control are computed on every draw. Intervals are pointwise
95% percentile intervals for the mean rate (`named`, `top5`) or mean reciprocal rank, conditional on
this model, lens, prompt and concept set. The **primary setting is fixed before the run**: the
protocol's span (user turn) and band 16–40, the best swap band in `concept_swap.ipynb` (other data),
at every strength. All other spans and bands are exploratory, and no setting is chosen by its result.
An all-zero arm gets `status="degenerate"` and no interval, which does not mean the rate is zero.

**Batching.** All trials share one prompt, so a batch is the same token row repeated with no padding.
For each (span, band), the (mode, concept, strength) trials run `BATCH_SIZE` per forward pass, each row
with its own direction and strength. Only the last position is unembedded. Results are cached to CSV
with the run settings, so re-executing the notebook does not repeat the sweep.

**Smoke mode.** `JLENS_SMOKE=1` swaps in a randomly initialized 4-layer GPT-2 and a random lens to
check the code on CPU without downloading weights. Its numbers mean nothing; the committed outputs
must come from the real run.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
REPO_DIR = os.path.abspath("jacobian-lens-gpt2")

# Opened from a clone (notebooks/<model>/<experiment>/): use the repository root above the notebook.
local_repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "jlens" / "workspace.py").is_file()), None)
if local_repo is not None:
    REPO_DIR = str(local_repo)
elif os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.append(REPO_DIR)

In [ ]:
import json
from contextlib import nullcontext

import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
from IPython.display import display
from tqdm.auto import tqdm

import jlens
from gpt2 import GPT2LensModel
from jlens.bootstrap import introspection_bootstrap
from jlens.workspace import (
    InjectDirection,
    InjectDirections,
    final_norm_centers,
    lens_directions,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)

In [ ]:
SMOKE = os.environ.get("JLENS_SMOKE") == "1"
MODEL_ID = "openai-community/gpt2-xl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTOCAST = DEVICE == "cuda"  # fp16 autocast, weights stay fp32 (as for the lens fit); checked in §1
LENS_PATH = Path(os.environ.get("JLENS_LENS_PATH",
                                Path(REPO_DIR) / "runs/gpt2-xl-convergence/snapshot_lens_134.pt"))
RUN_DIR = Path(os.environ.get("JLENS_RUN_DIR", Path(REPO_DIR) / "runs/gpt2-xl-introspection"))
DATASET = Path(REPO_DIR) / "data/experiments/verbal-introspection-gpt2.json"
PREFILL_KEY = "word"
BANDS = ["8-16", "16-24", "24-32", "32-40", "40-47", "16-40"]  # half-open block-output ranges
DEMO_BAND = "16-40"  # the best swap band in concept_swap.ipynb
STRENGTHS = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0]  # 0.0 is the shared control pass
MODES = ["J", "logit", "random"]
# Trials per forward pass (~130 tokens each); 256 adds a few GB to the 6 GB of fp32 weights.
BATCH_SIZE = int(os.environ.get("JLENS_BATCH_SIZE", "256"))
PRIMARY_SPAN = "user turn"  # the protocol's span; with DEMO_BAND, the pre-registered primary setting
BOOTSTRAP_OPTIONS = dict(n_resamples=int(os.environ.get("JLENS_BOOTSTRAP_RESAMPLES", "10000")),
                         confidence=0.95, seed=0, batch_size=256)
if SMOKE:
    DEVICE, AUTOCAST = "cpu", False
    RUN_DIR = Path(os.environ.get("JLENS_RUN_DIR", "/tmp/jlens-introspection-smoke"))
    BANDS, DEMO_BAND, BATCH_SIZE = ["1-3", "0-3"], "0-3", 64
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = RUN_DIR / "introspection_trials.csv"
INFO_PATH = RUN_DIR / "introspection_trials.json"
torch.manual_seed(0)
print(f"device={DEVICE} autocast={AUTOCAST} smoke={SMOKE} run_dir={RUN_DIR}")

In [ ]:
if SMOKE:
    tokenizer = transformers.AutoTokenizer.from_pretrained("openai-community/gpt2")
    config = transformers.GPT2Config(n_layer=4, n_embd=64, n_head=4)
    model = GPT2LensModel(transformers.GPT2LMHeadModel(config), tokenizer)
    generator = torch.Generator().manual_seed(0)
    lens = jlens.JacobianLens(
        jacobians={layer: torch.eye(64) + 0.1 * torch.randn(64, 64, generator=generator)
                   for layer in range(model.n_layers - 1)},
        n_prompts=1, d_model=64)
else:
    model = GPT2LensModel.from_pretrained(MODEL_ID, device=DEVICE)
    tokenizer = model.tokenizer
    lens = jlens.JacobianLens.load(str(LENS_PATH))
CENTERS = final_norm_centers(model)
assert CENTERS, "GPT-2 uses LayerNorm"
assert lens.d_model == model.d_model and lens.source_layers == list(range(model.n_layers - 1))
print(f"{model}  lens: {lens.n_prompts} prompts, layers {lens.source_layers[0]}-{lens.source_layers[-1]}")

## 1. The prompt, the injected spans and the concepts

The question turn is located by character offsets and decoded back, to confirm the span is the
question and not the role labels.

In [ ]:
spec = json.loads(DATASET.read_text())
PREFILL = spec["prefills"][PREFILL_KEY]
ROLE = {"user": "User:", "assistant": "Assistant:"}
# intro_prompt ends with an empty assistant turn: the prefill takes its place.
PROMPT = "\n\n".join(ROLE[m["role"]] + m["content"] for m in spec["intro_prompt"][:-1])
PROMPT += "\n\n" + ROLE["assistant"] + PREFILL
input_ids = model.encode(PROMPT)
encoded = tokenizer(PROMPT, return_offsets_mapping=True)
assert input_ids[0, 1:].tolist() == encoded.input_ids  # BOS + the plain tokenization

question = spec["intro_prompt"][2]["content"].strip()
start_char = PROMPT.index(question)
covered = [i + 1 for i, (a, b) in enumerate(encoded.offset_mapping)  # +1: BOS
           if a < start_char + len(question) and b > start_char]
SPANS = {"user turn": slice(covered[0], covered[-1] + 1),
         "whole prompt": slice(1, input_ids.shape[1])}  # BOS is an attention sink, never patched
print(PROMPT, end="\n\n")
print(f"{input_ids.shape[1]} tokens; user turn {SPANS['user turn']}:",
      repr(tokenizer.decode(input_ids[0, SPANS["user turn"]])))
print("last tokens:", [tokenizer.decode([i]) for i in input_ids[0, -4:].tolist()])

In [ ]:
concepts = pd.DataFrame(spec["concepts"])
ids = concepts.surface.map(lambda w: tokenizer.encode(w))
assert ids.str.len().eq(1).all() and PREFILL.endswith('"'), "rebuild the concept set for this tokenizer"
concepts["token_id"] = ids.str[0]
concept_ids = torch.tensor(concepts.token_id.values, device=DEVICE)
print(len(concepts), "concepts:", concepts.origin.value_counts().to_dict())

### Precision and the no-injection baseline

The control pass in fp32 and under fp16 autocast: the readout ranks of the concepts must agree
before the sweep is run in fp16.

In [ ]:
def autocast():
    return torch.autocast("cuda", dtype=torch.float16) if AUTOCAST else nullcontext()


@torch.no_grad()
def last_logits(ids):
    hidden = model.forward(ids).last_hidden_state[:, -1]  # ln_f already applied
    return model._lm_head(hidden).float()


def ranks_of(logits, targets):
    # Rank 1 = argmax; ties count in the target's favour, as in the Qwen notebook.
    return (logits > logits.gather(-1, targets[:, None])).sum(-1) + 1


control_fp32 = last_logits(input_ids)[0]
with autocast():
    control = last_logits(input_ids)[0]
control_ranks = ranks_of(control.expand(len(concepts), -1), concept_ids).cpu()
fp32_ranks = ranks_of(control_fp32.expand(len(concepts), -1), concept_ids).cpu()
print("top-5 with nothing injected:", [tokenizer.decode([i]) for i in control.topk(5).indices.tolist()])
print(f"fp32 vs autocast: same top-1 {bool(control.argmax() == control_fp32.argmax())}, "
      f"max |logit diff| {float((control - control_fp32).abs().max()):.3f}, "
      f"concept-rank Spearman {pd.Series(control_ranks).rank().corr(pd.Series(fp32_ranks).rank()):.4f}")
print(f"control: median concept rank {control_ranks.median():.0f}, named {int((control_ranks == 1).sum())}")

## 2. Directions

`[mode, layer, concept, d_model]` for every lens layer, built once on the device. Each row is
unit-normalized inside `InjectDirections`, so only the direction matters.

In [ ]:
LAYERS = lens.source_layers
with torch.no_grad():
    j_dirs = torch.stack([lens_directions(model, lens, concept_ids, layer, centers=CENTERS)
                          for layer in LAYERS])
    logit_dirs = lens_directions(model, None, concept_ids, 0, mode="logit", centers=CENTERS)
    random_dirs = torch.randn(j_dirs.shape, generator=torch.Generator().manual_seed(0)).to(DEVICE)
DIRECTIONS = torch.stack([j_dirs, logit_dirs.expand_as(j_dirs), random_dirs])  # MODES order
print(tuple(DIRECTIONS.shape), f"{DIRECTIONS.numel() * 4 / 2**20:.0f} MiB")
cos = torch.nn.functional.cosine_similarity(j_dirs, logit_dirs[None], dim=-1)
print("cosine(J, logit) by layer, mean over concepts:", cos.mean(-1)[::6].round(decimals=2).tolist())

## 3. What the model says, with and without an injection

Greedy 10-token continuations for three concepts. The prompt is re-run in full at each step (no KV
cache), and only prompt positions are injected, so generated tokens are never patched.

In [ ]:
def band_layers(band):
    start, stop = map(int, band.split("-"))
    return list(range(start, stop))


def inject(span, band, modes, concepts_idx, strengths):
    layers = band_layers(band)
    modes = torch.tensor(modes, device=DEVICE)
    concepts_idx = torch.tensor(concepts_idx, device=DEVICE)
    directions = {layer: DIRECTIONS[modes, layer, concepts_idx] for layer in layers}
    return InjectDirections(model, layers, directions, torch.tensor(strengths, device=DEVICE),
                            positions=SPANS[span])


@torch.no_grad()
def greedy(span, band, modes, concepts_idx, strengths, new_tokens=10):
    ids = input_ids.expand(len(strengths), -1)
    with inject(span, band, modes, concepts_idx, strengths), autocast():
        for _ in range(new_tokens):
            ids = torch.cat([ids, last_logits(ids).argmax(-1, keepdim=True)], dim=1)
    return [tokenizer.decode(row[input_ids.shape[1]:]) for row in ids.tolist()]


# The batched hook must match one InjectDirection pass per trial on this model.
check_layers = band_layers(DEMO_BAND)
with torch.no_grad(), inject("user turn", DEMO_BAND, [0, 1], [0, 1], [1.0, 2.0]), autocast():
    batched = last_logits(input_ids.expand(2, -1))
for row, (mode, strength) in enumerate([("J", 1.0), ("logit", 2.0)]):
    with torch.no_grad(), InjectDirection(model, lens, check_layers, int(concept_ids[row]), strength=strength,
                                          mode=mode, positions=SPANS["user turn"], centers=CENTERS), autocast():
        single = last_logits(input_ids)[0]
    assert torch.allclose(batched[row], single, atol=0.05 if AUTOCAST else 1e-3), (mode, (batched[row] - single).abs().max())
print("batched injection matches InjectDirection")

demo = pd.DataFrame([(span, c, s) for span in SPANS for c in [0, 1, 2] for s in (0.0, 1.0, 4.0)],
                    columns=["span", "concept_idx", "strength"])
demo["continuation"] = None
for span, group in demo.groupby("span", sort=False):
    demo.loc[group.index, "continuation"] = greedy(span, DEMO_BAND, [0] * len(group),
                                                   group.concept_idx.values, group.strength.values)
demo["concept"] = concepts.surface.values[demo.concept_idx]
demo["continuation"] = ('"' + demo.continuation).map(repr)
display(demo.set_index(["span", "concept", "strength"])[["continuation"]])

## 4. Sweep: all concepts, strengths, modes, bands and spans

In [ ]:
RUN_INFO = {"model_id": MODEL_ID, "smoke": SMOKE, "lens": str(LENS_PATH), "lens_n_prompts": lens.n_prompts,
            "autocast_fp16": AUTOCAST, "dataset": DATASET.name, "prefill": PREFILL_KEY, "bands": BANDS,
            "strengths": STRENGTHS, "modes": MODES, "spans": {k: [v.start, v.stop] for k, v in SPANS.items()},
            "random_seed": 0, "concepts": concepts.surface.tolist()}
trials = pd.DataFrame([(m, c, s) for m in range(len(MODES)) for c in range(len(concepts)) for s in STRENGTHS],
                      columns=["mode_idx", "concept_idx", "strength"])
print(f"{len(trials)} trials per (span, band) × {len(SPANS) * len(BANDS)} = "
      f"{len(trials) * len(SPANS) * len(BANDS)} injected trials, "
      f"{-(-len(trials) // BATCH_SIZE) * len(SPANS) * len(BANDS)} forward passes")

cached = INFO_PATH.exists() and OUT_PATH.exists() and json.loads(INFO_PATH.read_text()) == RUN_INFO
if cached:
    results = pd.read_csv(OUT_PATH)
    print("loaded cache")
else:
    targets_all = concept_ids[torch.tensor(trials.concept_idx.values, device=DEVICE)]
    frames = []
    for span in SPANS:
        for band in tqdm(BANDS, desc=span):
            ranks, top1 = [], []
            for start in range(0, len(trials), BATCH_SIZE):
                chunk = trials.iloc[start:start + BATCH_SIZE]
                with inject(span, band, chunk.mode_idx.values, chunk.concept_idx.values,
                            chunk.strength.values), autocast():
                    logits = last_logits(input_ids.expand(len(chunk), -1))
                ranks.append(ranks_of(logits, targets_all[start:start + len(chunk)]))
                top1.append(logits.argmax(-1))
            frame = trials.copy()
            frame["span"], frame["band"] = span, band
            frame["rank"] = torch.cat(ranks).cpu().numpy()  # one sync per (span, band)
            frame["top1_id"] = torch.cat(top1).cpu().numpy()
            frames.append(frame)
    injected = pd.concat(frames, ignore_index=True)
    # Strength 0 is the same clean pass for every span, band and mode.
    control_rows = pd.DataFrame([(m, c, 0.0, span, band, int(control_ranks[c]), int(control.argmax()))
                                 for span in SPANS for band in BANDS for m in range(len(MODES))
                                 for c in range(len(concepts))], columns=injected.columns)
    results = pd.concat([control_rows, injected], ignore_index=True)
    results["mode"] = [MODES[m] for m in results.mode_idx]
    results["concept"] = concepts.surface.values[results.concept_idx]
    results["origin"] = concepts.origin.values[results.concept_idx]
    results["top1"] = [tokenizer.decode([i]) for i in results.top1_id]
    results = results.drop(columns=["mode_idx", "concept_idx", "top1_id"])
    results.to_csv(OUT_PATH, index=False)
    INFO_PATH.write_text(json.dumps(RUN_INFO, indent=1) + "\n")
results["rr"] = 1.0 / results["rank"]
results["named"] = results["rank"].eq(1)
results["top5"] = results["rank"].le(5)
print(len(results), "trials")
results.head(3)

### Median reciprocal rank vs strength

The paper's figure. The `0.0` column is the no-injection control, shared by all modes.

In [ ]:
display(results.pivot_table(index=["span", "mode", "band"], columns="strength", values="rr",
                           aggfunc="median").round(4))

### Fraction of trials where the concept is named (rank 1) and in the top 5

In [ ]:
display(results.pivot_table(index=["span", "mode", "band"], columns="strength",
                           values="named", aggfunc="mean").round(3))
display(results.pivot_table(index=["span", "mode", "band"], columns="strength",
                           values="top5", aggfunc="mean").round(3))

In [ ]:
COLORS = {"J": "#2a78d6", "logit": "#eb6834", "random": "#52514e"}
fig, axes = plt.subplots(len(SPANS), 2, figsize=(13, 4.3 * len(SPANS)), squeeze=False)
for row, span in enumerate(SPANS):
    for mode, color in COLORS.items():
        picked = results[results["mode"].eq(mode) & results.band.eq(DEMO_BAND) & results.span.eq(span)]
        axes[row][0].plot(picked.groupby("strength").rr.median(), marker="o", color=color, label=f"{mode} directions")
        axes[row][1].plot(picked.groupby("strength").top5.mean(), marker="o", color=color, label=f"{mode} directions")
    for ax, label in zip(axes[row], ("median reciprocal rank", "fraction in top 5"), strict=True):
        ax.set_xlabel("injection strength")
        ax.set_ylabel(label)
        ax.set_xscale("symlog", linthresh=0.125)
        ax.set_title(f"injected into: {span}")
        ax.grid(alpha=0.3)
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"GPT-2 XL: injected-thought report, band {DEMO_BAND}, {len(concepts)} concepts")
fig.tight_layout()
plt.show()

### Does the control stay quiet?

In [ ]:
ctrl = results[results.strength.eq(0.0)].drop_duplicates("concept")
print(f"control: named {ctrl.named.mean():.3f}, top-5 {ctrl.top5.mean():.3f}, "
      f"median rank {ctrl['rank'].median():.0f}")
display(ctrl.groupby("origin")["rank"].describe()[["count", "25%", "50%", "75%"]])
print("top-1 at the quote with nothing injected:", repr(ctrl.top1.iloc[0]))

### Which concepts are reported, and what the model says instead

The J arm in the primary span and band at its strongest injection.

In [ ]:
picked = results[results.span.eq(PRIMARY_SPAN) & results["mode"].eq("J") & results.band.eq(DEMO_BAND)
                 & results.strength.eq(max(STRENGTHS))].set_index("concept")
display(picked[["origin", "rank", "top1"]].sort_values("rank").head(10))
display(picked.top1.value_counts().head(8).to_frame("top-1 count"))

## 5. Paired bootstrap intervals

Concept-clustered, with the three direction arms and the no-injection control of the same concept in
every draw. Computed from the cached trials; changing the bootstrap settings needs no model pass.

In [ ]:
ci_table = introspection_bootstrap(results, **BOOTSTRAP_OPTIONS)
ci_table["analysis_role"] = "exploratory"
ci_table.loc[ci_table.span.eq(PRIMARY_SPAN) & ci_table.band.eq(DEMO_BAND), "analysis_role"] = "primary"

# The bootstrap keeps the plain equal-concept means as its point estimates.
means = results[results.strength.gt(0)].groupby(["span", "band", "strength", "mode"])[["named", "top5", "rr"]].mean()
arm_rows = ci_table[ci_table.statistic.eq("arm") & ci_table.arm.ne("control")]
for row in arm_rows.itertuples():
    assert abs(row.estimate - means.loc[(row.span, row.band, row.strength, row.arm), row.metric]) < 1e-12
ci_table.to_csv(RUN_DIR / "introspection_bootstrap.csv", index=False)
print(ci_table.status.value_counts().to_dict())


def ci_view(table):
    label = table.arm.where(table.statistic.eq("arm"), table.arm + " − " + table.reference)
    fmt = "{:.3g}".format
    cell = (table.estimate.map(fmt) + " [" + table.ci_low.map(fmt) + ", " + table.ci_high.map(fmt) + "]").where(
        table.status.eq("ok"), table.estimate.map(fmt) + " (" + table.status + ")")
    return table.assign(label=label, cell=cell).pivot(index="strength", columns="label", values="cell")


primary = ci_table[ci_table.analysis_role.eq("primary")]
for metric in ["named", "top5", "rr"]:
    print(f"primary setting ({PRIMARY_SPAN}, band {DEMO_BAND}): {metric}, "
          f"{100 * BOOTSTRAP_OPTIONS['confidence']:g}% CI, n={int(primary.n_units.iloc[0])} concepts")
    display(ci_view(primary[primary.metric.eq(metric)]))

In [ ]:
ARM_COLORS = {**COLORS, "control": "#9a9a9a"}
fig, axes = plt.subplots(len(SPANS), 2, figsize=(13, 4.3 * len(SPANS)), squeeze=False)
for row, span in enumerate(SPANS):
    for column, metric in enumerate(["named", "top5"]):
        ax = axes[row][column]
        table = ci_table[ci_table.span.eq(span) & ci_table.band.eq(DEMO_BAND) & ci_table.metric.eq(metric)
                         & ci_table.statistic.eq("arm")]
        for arm, color in ARM_COLORS.items():
            line = table[table.arm.eq(arm)].sort_values("strength")
            ok = line.status.eq("ok")
            ax.plot(line.strength, line.estimate, marker="o", color=color, label=arm)
            ax.fill_between(line.strength, line.ci_low.where(ok, line.estimate),
                            line.ci_high.where(ok, line.estimate), color=color, alpha=0.15)
        ax.set_xscale("log", base=2)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel("injection strength")
        ax.set_ylabel({"named": "fraction named (rank 1)", "top5": "fraction in top 5"}[metric])
        role = "primary" if span == PRIMARY_SPAN else "contrast, exploratory"
        ax.set_title(f"injected into: {span} ({role})")
        ax.grid(alpha=0.3)
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"GPT-2 XL, band {DEMO_BAND}: mean over {len(concepts)} concepts with pointwise "
             f"{100 * BOOTSTRAP_OPTIONS['confidence']:g}% concept-bootstrap intervals")
fig.tight_layout()
plt.show()

### Paired differences in the primary setting

J − random is the specificity of the J-lens direction over a perturbation of the same size, and
J − control is the effect of injecting at all. Both must exclude zero for a positive claim.

In [ ]:
diffs = primary[primary.statistic.eq("difference") & primary.metric.isin(["named", "top5"])]
display(diffs.pivot_table(index=["metric", "strength"], columns=["arm", "reference"],
                          values=["estimate", "ci_low", "ci_high"], dropna=False).round(3))

### By concept origin

Kept originals vs added nouns, in the primary setting, each bootstrapped over its own concepts.

In [ ]:
primary_results = results[results.span.eq(PRIMARY_SPAN) & results.band.eq(DEMO_BAND)]
by_origin = introspection_bootstrap(primary_results, keys=["span", "band", "strength", "origin"],
                                    **BOOTSTRAP_OPTIONS)
picked = by_origin[by_origin.metric.eq("top5") & by_origin.statistic.eq("arm")]
display(picked.pivot_table(index=["origin", "strength"], columns="arm",
                           values=["estimate", "ci_low", "ci_high"], dropna=False).round(3))

## 6. Conclusions

PENDING (filled in after the run on the V100).